# 시행계획 시도 분리·정제 공통 유틸 사용 템플릿

이 파일을 복사한 뒤 파일명을 `YYYYMMDD_EDA_{연도}_전국_시도분리정제.ipynb` 형식으로 변경한다.

복사 후 반드시 확인할 항목:

1. `YEAR`, `SOURCE_FILE`, `SOURCE_SHEET`
2. 해당 연도의 0원 표기인 `ZERO_BUDGET_TOKENS`
3. 연도 고유 반복 머리글인 `EXTRA_HEADER_PATTERNS` — 2016~2020(제3차 기본계획) 원본은 시도별 단위표기 헤더 행("(단위 : 백만원)" 등)이 있는지 반드시 확인하고, 있으면 `UNIT_NOTATION_PATTERN`을 추가한다 (`src/features/pipeline_common.py` 참고, 이슈 #24)
4. **재원구분 이중계상 여부 `NEEDS_FUNDING_SOURCE_CLEANUP`** — 2016~2019(제3차 기본계획) 원본은 세부사업 하나가 계/국비/지방비(도비·시군비·비예산 등)로 최대 3~4행 중복 표기돼 있다. 원본 `사업분류재정구분` 값이 "계"/"국비"/"지방비" 등인지 확인 후 True로 변경한다 (`drop_exact_duplicate_rows`, `select_total_budget_rows`, 이슈 #26)
5. 소계 QA 허용오차인 `QA_TOLERANCE`
6. LLM 실행 여부와 연도별 체크포인트 경로

> 원본 예산 컬럼은 덮어쓰지 않는다. LLM 셀은 체크포인트와 API 키를 확인하기 전 실행하지 않는다.

In [1]:
import hashlib
import os
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.features.llm_refine import (  # noqa: E402
    call_llm_once,
    needs_llm_rerun,
    numbers_preserved,
    preservation_violations,
    run_checkpointed_refinement,
)
from src.features.pipeline_common import (  # noqa: E402
    UNIT_NOTATION_PATTERN,
    assign_labels,
    build_subtotal_qa,
    calculate_budget_changes,
    classify_row,
    clean_text,
    drop_exact_duplicate_rows,
    get_sido_dir,
    normalize_budget_type,
    select_total_budget_rows,
    to_numeric_budget,
)
from src.features.text_match import (  # noqa: E402
    check_pattern,
    check_pattern_broad,
    dedup_label,
    extract_via_regex,
    find_near_duplicates,
)

print(f"프로젝트 루트: {PROJECT_ROOT}")

프로젝트 루트: D:\University\yumocha\yumocha-review-2020


## 1. 연도·입력·예외 설정

아래 셀은 복사한 노트북에서 가장 먼저 수정한다. `비예산`, `·` 등을 0으로 처리할지는 해당 연도 원본을 확인한 뒤 결정한다.

In [2]:
YEAR = 2020  # 시행계획 연도
PREVIOUS_YEAR = YEAR - 1

CURRENT_BUDGET_COL = f"{YEAR}년 예산"
PREVIOUS_BUDGET_COL = f"{PREVIOUS_YEAR}년 예산"
CURRENT_NUM_COL = f"{YEAR}년_예산_num"
PREVIOUS_NUM_COL = f"{PREVIOUS_YEAR}년_예산_num"

DATA_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
REPORTS_DIR = PROJECT_ROOT / "reports"
CONFIG_DIR = PROJECT_ROOT / "configs"

SOURCE_FILE = (
    DATA_DIR / "칼럼정렬" / "세부사업표추출_2020년도 지방자치단체 저출산고령사회 시행계획 (제3차 기본계획)_칼럼정렬.xlsx"
)
SOURCE_SHEET = "정리본_자동"
TABLE1_FILE = SOURCE_FILE

ZERO_BUDGET_TOKENS = ("-",)
EXTRA_HEADER_PATTERNS = (UNIT_NOTATION_PATTERN,)
NEEDS_FUNDING_SOURCE_CLEANUP = False
QA_TOLERANCE = 0.01
RUN_LLM = os.getenv("RUN_LLM", "true").strip().lower() in {"1", "true", "yes"}

CHECKPOINT_PATH = INTERIM_DIR / f"{YEAR}_llm_정제_체크포인트.csv"
QA_PATH = REPORTS_DIR / f"{YEAR}_전국_QA_검증결과.csv"
ROW_TYPE_REVIEW_PATH = REPORTS_DIR / f"{YEAR}_행유형_후보_검토.csv"
LABEL_PROPAGATION_PATH = REPORTS_DIR / f"{YEAR}_경남_7494_라벨전파_검증.csv"
LLM_SUMMARY_PATH = REPORTS_DIR / f"{YEAR}_LLM_실행요약.csv"
PRESERVATION_REVIEW_PATH = REPORTS_DIR / f"{YEAR}_LLM_보존위반_검토.csv"


## 2. 데이터 로드와 기본 확인

In [3]:
if not SOURCE_FILE.exists():
    raise FileNotFoundError(f"SOURCE_FILE을 실제 파일로 수정하세요: {SOURCE_FILE}")

SOURCE_SHA256_BEFORE = hashlib.sha256(SOURCE_FILE.read_bytes()).hexdigest()
df_raw = pd.read_excel(SOURCE_FILE, sheet_name=SOURCE_SHEET)

required_columns = {
    "지역",
    "세부사업명",
    "사업분류재정구분",
    CURRENT_BUDGET_COL,
    PREVIOUS_BUDGET_COL,
    "주요내용",
    "원본행",
}
missing_columns = required_columns.difference(df_raw.columns)
if missing_columns:
    raise KeyError(f"입력 파일에 필요한 컬럼이 없습니다: {sorted(missing_columns)}")

expected_shape = (7003, 12)
if df_raw.shape != expected_shape:
    raise ValueError(f"입력 크기가 예상과 다릅니다: {df_raw.shape} != {expected_shape}")

region_count = df_raw["지역"].nunique()
if region_count != 17:
    raise ValueError(f"지역 수가 예상과 다릅니다: {region_count} != 17")

finance_values = set(
    df_raw["사업분류재정구분"].dropna().astype("string").str.strip().unique()
)
if finance_values != {"공통", "자체"}:
    raise ValueError(f"예상하지 못한 사업분류재정구분 값: {sorted(finance_values)}")

print("데이터 크기:", df_raw.shape)
print("지역 수:", region_count)
print("사업분류재정구분:", sorted(finance_values))
display(df_raw.head())

데이터 크기: (7003, 12)
지역 수: 17
사업분류재정구분: ['공통', '자체']


,지역,세부사업명,사업분류재정구분,2020년 예산,2019년 예산,증감액,비율,주요내용,원본표구간,머리글행,원본행,행유형
0,서울,(단위 : 백만원),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,서울,Ⅰ. 공통사업,NaN,6179699,5809008,370691,6.4,NaN,1.0,3.0,5.0,데이터행
2,서울,1. 함께 돌보고 함께 일하는 사회(공통사업),NaN,3164917,3131359,33558,1.1,NaN,1.0,3.0,6.0,데이터행
3,서울,청년 주거지원 강화 - 다양한 청년주택 공급 확대(역세권청년주택),공통,26659,33301,-6642,-19.9,지원대상：청년 및 신혼부부 지원내용：역세권 민간소유 토지를 개발 하여 청년 및 신혼...,1.0,3.0,7.0,데이터행
4,서울,여성일자리 창출,공통,38593,33409,5184,15.5,"여성일자리창출：39,305명",1.0,3.0,8.0,데이터행


## 3. 행 분류·계층 전파·숫자 변환

소계 QA에 원본 소계 행의 숫자값도 필요하므로 `df_labeled` 전체에 숫자 컬럼을 만든다.

In [4]:
if NEEDS_FUNDING_SOURCE_CLEANUP:
    df_raw = drop_exact_duplicate_rows(df_raw)
    df_raw = select_total_budget_rows(
        df_raw,
        budget_cols=[CURRENT_BUDGET_COL, PREVIOUS_BUDGET_COL],
        zero_tokens=ZERO_BUDGET_TOKENS,
    )
    print("재원구분(계/국비/지방비) 정리 후 행 수:", len(df_raw))

df_raw["사업행구분"] = df_raw["세부사업명"].apply(
    lambda value: classify_row(value, extra_header_patterns=EXTRA_HEADER_PATTERNS)
)

# 2020년 중분류 제목은 공통 함수가 인식하는 "(공통)/(자체)"가 아니라
# "(공통사업)/(자체사업)"으로 끝나므로 이 노트북에서만 보정한다.
detail_name = df_raw["세부사업명"].astype("string").str.strip()
medium_category_mask = detail_name.str.match(r"^\d+\.", na=False) & detail_name.str.contains(
    r"\((?:공통|자체)사업\)$",
    regex=True,
    na=False,
)

# 경기의 도·시/도비·시비 보조 소계는 상위 중분류 QA에서 중복 집계하지 않도록 제외한다.
auxiliary_subtotal_mask = detail_name.str.match(r"^\d+\.", na=False) & detail_name.str.contains(
    r"\((?:도|시|도비|시비)\s*자체사업\)$",
    regex=True,
    na=False,
)

# 인천·세종·경기의 전체 총계는 세부사업이 아니다.
compact_detail_name = detail_name.str.replace(r"\s+", "", regex=True)
grand_total_mask = compact_detail_name.isin({"총계", "총계(공통사업+자체사업)", "합계"})

# 경남 원본행 7494는 "3. 인구구조 변화..."로 적혀 있으나 뒤따르는 행은 노후사업이고,
# 실제 3번 중분류 소계가 원본행 7651에 다시 나온다. 반복된 오기 머리글로 제외한다.
gyeongnam_mislabelled_heading_mask = df_raw["지역"].eq("경남") & df_raw["원본행"].eq(7494)

df_raw.loc[medium_category_mask, "사업행구분"] = "중분류_소계"
df_raw.loc[
    auxiliary_subtotal_mask | grand_total_mask | gyeongnam_mislabelled_heading_mask,
    "사업행구분",
] = "헤더반복"

# 원본 소계는 보존하고, QA 비교에만 원본행 7494의 잘못 포함된 29,994백만원을 제외한다.
gyeongnam_old_age_label = "2. 함께 만들어가는 행복한 노후(자체사업)"
gyeongnam_population_label = "3. 인구구조 변화 적극 대비 (자체사업)"
gyeongnam_old_age_subtotal_mask = (
    df_raw["지역"].eq("경남")
    & detail_name.eq(gyeongnam_old_age_label)
    & df_raw["사업행구분"].eq("중분류_소계")
)
if int(gyeongnam_old_age_subtotal_mask.sum()) != 1:
    raise ValueError("경남 노후사업 원본 소계 행이 정확히 1건이어야 합니다.")

df_raw["QA_소계_보정액"] = 0.0
df_raw["QA_소계_보정근거"] = ""
df_raw.loc[gyeongnam_old_age_subtotal_mask, "QA_소계_보정액"] = -29994.0
df_raw.loc[gyeongnam_old_age_subtotal_mask, "QA_소계_보정근거"] = (
    "원본행 7494에 잘못 포함된 29,994백만원을 QA 비교 소계에서 제외"
)

row_type_review_mask = (
    detail_name.str.match(UNIT_NOTATION_PATTERN, na=False)
    | auxiliary_subtotal_mask
    | grand_total_mask
    | gyeongnam_mislabelled_heading_mask
)
row_type_review = df_raw.loc[
    row_type_review_mask,
    ["지역", "원본행", "세부사업명", "사업행구분"],
].copy()
row_type_review["판정"] = "제외"
row_type_review["판정근거"] = ""
row_type_review.loc[
    detail_name.str.match(UNIT_NOTATION_PATTERN, na=False), "판정근거"
] = "단위표기 머리글"
row_type_review.loc[auxiliary_subtotal_mask, "판정근거"] = (
    "경기 도·시/도비·시비 보조 소계"
)
row_type_review.loc[grand_total_mask, "판정근거"] = "지역 전체 총계"
row_type_review.loc[gyeongnam_mislabelled_heading_mask, "판정근거"] = (
    "뒤따르는 노후사업 및 원본행 7651의 실제 3번 소계와 중복되는 원본 오기 머리글"
)

if len(row_type_review) != 11 or row_type_review["판정근거"].eq("").any():
    raise ValueError("2020년 행 유형 예외 11건을 다시 확인하세요.")

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
row_type_review.to_csv(ROW_TYPE_REVIEW_PATH, index=False, encoding="utf-8-sig")
print(f"행 유형 검토표 저장 완료: {ROW_TYPE_REVIEW_PATH}")


df_labeled = df_raw.groupby("지역", group_keys=False).apply(assign_labels)
df_labeled["지역"] = df_raw.loc[df_labeled.index, "지역"]
df_labeled[CURRENT_NUM_COL] = to_numeric_budget(
    df_labeled[CURRENT_BUDGET_COL], zero_tokens=ZERO_BUDGET_TOKENS
)
df_labeled[PREVIOUS_NUM_COL] = to_numeric_budget(
    df_labeled[PREVIOUS_BUDGET_COL], zero_tokens=ZERO_BUDGET_TOKENS
)

# 7494 오기 머리글이 뒤 노후사업의 중분류를 3번으로 바꾸지 않았는지 행 단위로 검증한다.
gyeongnam_rows = df_labeled["지역"].eq("경남")
post_7494_leaf_mask = (
    gyeongnam_rows
    & df_labeled["원본행"].gt(7494)
    & df_labeled["원본행"].lt(7651)
    & df_labeled["사업행구분"].eq("세부사업")
)
label_check_mask = (
    post_7494_leaf_mask
    | (gyeongnam_rows & df_labeled["원본행"].isin([7494, 7651]))
)
label_check = df_labeled.loc[
    label_check_mask,
    ["지역", "원본행", "세부사업명", "사업행구분", "중분류"],
].copy()
label_check["기대행구분"] = "세부사업"
label_check["기대중분류"] = gyeongnam_old_age_label
label_check.loc[label_check["원본행"].eq(7494), "기대행구분"] = "헤더반복"
label_check.loc[label_check["원본행"].eq(7651), "기대행구분"] = "중분류_소계"
label_check.loc[label_check["원본행"].eq(7651), "기대중분류"] = (
    gyeongnam_population_label
)
label_check["통과"] = (
    label_check["사업행구분"].eq(label_check["기대행구분"])
    & label_check["중분류"].eq(label_check["기대중분류"])
)
if label_check.empty or not label_check["통과"].all():
    raise ValueError("경남 7494 뒤 중분류 라벨 전파 검증에 실패했습니다.")
if df_labeled.loc[gyeongnam_mislabelled_heading_mask, CURRENT_NUM_COL].item() != 29994.0:
    raise ValueError("경남 원본행 7494의 보정 대상 금액이 29,994가 아닙니다.")
if df_labeled.loc[gyeongnam_old_age_subtotal_mask, CURRENT_NUM_COL].item() != 87048.0:
    raise ValueError("경남 노후사업 원본 소계 87,048이 보존되지 않았습니다.")
label_check.to_csv(LABEL_PROPAGATION_PATH, index=False, encoding="utf-8-sig")
print("경남 7494 라벨 전파 검증:", len(label_check), "행 통과")

df_leaf = df_labeled[df_labeled["사업행구분"] == "세부사업"].copy()
print(df_labeled["사업행구분"].value_counts())
print("세부사업 행 수:", len(df_leaf))

행 유형 검토표 저장 완료: D:\University\yumocha\yumocha-review-2020\reports\2020_행유형_후보_검토.csv
경남 7494 라벨 전파 검증: 142 행 통과
사업행구분
세부사업      6858
중분류_소계     100
대분류_소계      34
헤더반복        11
Name: count, dtype: int64
세부사업 행 수: 6858


## 4. 텍스트 정제와 예산 재계산

In [5]:
for column in ["세부사업명", "대분류", "중분류"]:
    df_leaf[column] = clean_text(df_leaf[column])

df_leaf["주요내용"] = clean_text(df_leaf["주요내용"], strip_leading_bullet=True)
df_leaf["사업분류재정구분"] = normalize_budget_type(df_leaf["사업분류재정구분"])

budget_changes = calculate_budget_changes(
    df_leaf[CURRENT_NUM_COL],
    df_leaf[PREVIOUS_NUM_COL],
)
df_leaf[["당해예산", "전년도예산", "증감액", "증감율"]] = budget_changes
display(df_leaf[["지역", "세부사업명", "당해예산", "전년도예산", "증감액", "증감율"]].head())

,지역,세부사업명,당해예산,전년도예산,증감액,증감율
3137,강원,아이돌봄서비스 지원,22680.0,21515.0,1165.0,5.4
3138,강원,공동육아나눔터 운영,781.0,594.0,187.0,31.5
3139,강원,한부모가족자녀 양육비 등 지원,11666.0,11678.0,-12.0,-0.1
3140,강원,권역별 미혼모부자 거점기관 운영,51.0,50.0,1.0,2.0
3141,강원,청소년한부모 자립지원,181.0,181.0,0.0,0.0


## 5. 중분류 소계 QA와 저장

QA 계산은 공통 함수가 담당하고, 연도별 파일 저장은 노트북에서 담당한다.

In [6]:
qa = build_subtotal_qa(
    df_labeled,
    budget_col=CURRENT_NUM_COL,
    tolerance=QA_TOLERANCE,
    subtotal_adjustment_col="QA_소계_보정액",
    subtotal_adjustment_reason_col="QA_소계_보정근거",
)

qa["원인분류"] = "판정불가"
qa.loc[qa["결과"].eq("일치"), "원인분류"] = "일치"
qa.loc[
    qa["결과"].eq("불일치") & qa["허용기준결과"].eq("허용"),
    "원인분류",
] = "허용범위 내 차이"
qa.loc[qa["허용기준결과"].eq("초과"), "원인분류"] = "원본 소계 불일치"
qa["판정근거"] = ""

ulsan_common_mask = qa["지역"].eq("울산") & qa["중분류"].eq(
    "1. 함께 돌보고 함께 일하는 사회(공통사업)"
)
qa.loc[ulsan_common_mask, "판정근거"] = (
    "원본 소계가 첫 5개 세부사업 합계 83,577백만원을 누락"
)

ulsan_own_mask = qa["지역"].eq("울산") & qa["중분류"].eq(
    "3. 인구구조 변화 적극 대비 (자체사업)"
)
qa.loc[ulsan_own_mask, "판정근거"] = (
    "원본 소계가 다둥이 행복렌트카 지원 50백만원을 누락"
)

gyeongnam_own_mask = qa["지역"].eq("경남") & qa["중분류"].eq(
    gyeongnam_old_age_label
)
qa.loc[gyeongnam_own_mask, "판정근거"] = (
    "원본 소계에서 29,994백만원을 보정한 비교 소계와 leaf 합계의 차이 +1백만원"
)

qa_over = qa[qa["허용기준결과"].eq("초과")]
expected_cause_counts = {"일치": 67, "허용범위 내 차이": 31, "원본 소계 불일치": 2}
expected_allowance_counts = {"허용": 98, "초과": 2}
actual_cause_counts = qa["원인분류"].value_counts().to_dict()
actual_allowance_counts = qa["허용기준결과"].value_counts().to_dict()
expected_over_groups = {
    ("울산", "1. 함께 돌보고 함께 일하는 사회(공통사업)"),
    ("울산", "3. 인구구조 변화 적극 대비 (자체사업)"),
}
actual_over_groups = set(qa_over[["지역", "중분류"]].itertuples(index=False, name=None))
if (
    len(qa) != 100
    or actual_cause_counts != expected_cause_counts
    or actual_allowance_counts != expected_allowance_counts
    or actual_over_groups != expected_over_groups
    or qa_over["판정근거"].eq("").any()
):
    raise ValueError(
        "2020년 QA 필수 수치가 달라졌습니다: "
        f"원인={actual_cause_counts}, 허용={actual_allowance_counts}, 초과={actual_over_groups}"
    )

gyeongnam_qa = qa.loc[gyeongnam_own_mask].squeeze()
expected_gyeongnam_values = {
    "원본_소계값": 87048.0,
    "보정액": -29994.0,
    "QA_비교_소계값": 57054.0,
    "leaf_합계": 57055.0,
    "차이": 1.0,
}
if any(gyeongnam_qa[column] != value for column, value in expected_gyeongnam_values.items()):
    raise ValueError("경남 노후사업 QA 보정 수치가 기준과 다릅니다.")
if (
    gyeongnam_qa["결과"] != "불일치"
    or gyeongnam_qa["허용기준결과"] != "허용"
    or gyeongnam_qa["원인분류"] != "허용범위 내 차이"
    or not str(gyeongnam_qa["보정근거"]).strip()
):
    raise ValueError("경남 노후사업이 허용범위 내 불일치로 판정되지 않았습니다.")

display(qa["원인분류"].value_counts(dropna=False))
display(qa_over.sort_values("오차율(%)", key=lambda series: series.abs(), ascending=False))

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
qa.to_csv(QA_PATH, index=False, encoding="utf-8-sig")
print(f"QA 저장 완료: {QA_PATH}")

원인분류
일치           67
허용범위 내 차이    31
원본 소계 불일치     2
Name: count, dtype: int64

,지역,대분류,중분류,원본_소계값,원본_소계출처,보정액,보정근거,QA_비교_소계값,leaf_합계,차이,절대차이,오차율(%),절대오차율(%),QA_병합상태,결과,허용기준결과,원인분류,판정근거
59,울산,Ⅰ. 공통사업,1. 함께 돌보고 함께 일하는 사회(공통사업),326481.0,중분류제목행,0.0,,326481.0,410058.0,83577.0,83577.0,25.6,25.6,양쪽존재,불일치,초과,원본 소계 불일치,"원본 소계가 첫 5개 세부사업 합계 83,577백만원을 누락"
64,울산,Ⅱ. 지자체 자체사업,3. 인구구조 변화 적극 대비 (자체사업),283.0,중분류제목행,0.0,,283.0,333.0,50.0,50.0,17.67,17.67,양쪽존재,불일치,초과,원본 소계 불일치,원본 소계가 다둥이 행복렌트카 지원 50백만원을 누락


QA 저장 완료: D:\University\yumocha\yumocha-review-2020\reports\2020_전국_QA_검증결과.csv


## 6. 주요내용 라벨 정리·부분 추출·유사 사업명 후보

In [7]:
df_leaf["주요내용"] = df_leaf["주요내용"].apply(dedup_label)
df_leaf["주요내용_패턴"] = df_leaf["주요내용"].apply(check_pattern)
df_leaf["주요내용_패턴_확장"] = df_leaf["주요내용"].apply(check_pattern_broad)

regex_extracted = pd.DataFrame(
    df_leaf["주요내용"].apply(extract_via_regex).tolist(),
    index=df_leaf.index,
)
df_leaf["지원대상"] = regex_extracted["지원대상"]
df_leaf["지원내용_상세"] = regex_extracted["지원내용"]

near_duplicates = find_near_duplicates(df_leaf, threshold=90)
print("유사 사업명 검토 후보:", len(near_duplicates))
display(near_duplicates.head(20))

유사 사업명 검토 후보: 250


,지역,중분류,세부사업명1,당해예산1,주요내용1,세부사업명2,당해예산2,주요내용2,유사도
0,울산,1. 함께 돌보고 함께 일하는 사회(공통사업),산모·신생아 건강관리서비스 지원대상 확대,3568.0,"지원대상 : 기준 중위소득 100%이하 출산 가정 지원내용 : 산모, 신생아 양육지...",산모신생아 건강관리서비스 지원대상 확대,4.0,지원대상 : 기준 중위소득 100%이하 출산 가정 지원내용 : 산모신생아 가정 방문...,97.674419
1,경기,1. 함께 돌보고 함께 일하는 사회(자체사업),"다자녀가정, 한부모가정 수도요금 감면",<NA>,"한부모가정, 만18세 이하 자녀가 3명이상 동일세대로 구성되어 있는 가구를 대상 사...","다자녀가정, 한부모가정 하수도요금 감면",<NA>,"한부모가정, 만18세 이하 자녀가 3명이상 동일세대로 구성되어 있는 가구를 대상 사...",97.560976
2,부산,1. 함께 돌보고 함께 일하는 사회(자체사업),정부지원 어린이집 차량운영비 지원,168.0,"정부지원 어린이집 차량운영비, 전세버스 임대차 경비 지원(280개소, 1개소 당 월...",정부미지원 어린이집 차량운영비 지원,72.0,사하구 소재 정부미지원 평가인증 어린 이집에 차량운영비 지원,97.297297
3,부산,1. 함께 돌보고 함께 일하는 사회(자체사업),정부지원 어린이집 차량운영비 지원,168.0,"정부지원 어린이집 차량운영비, 전세버스 임대차 경비 지원(280개소, 1개소 당 월...",정부미지원 어린이집 차량운영비 지원,9.0,사업내용 : 정부미지원어린이집 차량운영비 지원 사업대상 : 관내 어린이집 15개소,97.297297
4,충남,2. 함께 만들어가는 행복한 노후(공통사업),경로당 냉난방비 및 양곡비 지원,873.0,"지원대상 : 등록 403경로당 지원내용 : 난방비 연5개월*32만원, 냉방비 연2개...",경로당 냉난방비 및 양곡 지원,1116.0,경로당 난방비 및 양곡비 지원,96.969697
5,경남,2. 함께 만들어가는 행복한 노후(자체사업),4대 이상 가정 효도수당 지원,20.0,사천시에 3년 이상 주소지를 둔 4대 이상 가정에 설/명절 효도수당 각 50만원씩 지급,4세대 이상 가정 효도수당 지원,8.0,거창군에 주소를 둔 4대 이상 거주가정 매월 10만원 지원,96.969697
6,강원,2. 함께 만들어가는 행복한 노후(자체사업),저소득층 노인 건강보험료 지원,108.0,영월군 주소 최저보험료 이하 65세 이상 노인세대,저소득 노인 건강보험료 지원,78.0,저소득 노인가구 대상 건강보험료 지원,96.774194
7,전남,2. 함께 만들어가는 행복한 노후(자체사업),노인 목욕비 및 이미용비 지원,2390.0,어르신 이용권 지원,노인 목욕 및 이미용비 지원,7710.0,노인 목욕 및 이‧ 미용비 지원,96.774194
8,강원,2. 함께 만들어가는 행복한 노후(자체사업),저소득 노인 건강보험료 지원,77.0,저소득노인가구 건강보험료 지원,저소득층 노인 건강보험료 지원,108.0,영월군 주소 최저보험료 이하 65세 이상 노인세대,96.774194
9,경기,1. 함께 돌보고 함께 일하는 사회(자체사업),다자녀 가구 수도요금 감면,7.0,다자녀 가구 가정용 요금 10톤 감면,다자녀 가구 상수도요금 감면,0.0,"만18세 미만 3자녀 이상을 둔 가구에 대하 여 수도요금 감면(3자녀 20%, 4자...",96.551724


## 7. 원본 Table 1 대조(필요 시)

원본 파일의 컬럼 배치가 다르면 `column_indices`를 해당 연도에 맞게 변경한다.

In [8]:
# 예시: 실제 행 번호로 교체한 뒤 주석을 해제한다.
# df_table1 = pd.read_excel(TABLE1_FILE, sheet_name="Table 1", header=None)
# display(show_table1_around(df_table1, center_excel_row=100, window=3, label="원본 대조"))

## 8. LLM 보존형 교정

기본값은 `RUN_LLM=true`이며, API 호출 없는 사전 검증 실행은 환경변수 `RUN_LLM=false`를 사용한다. 체크포인트는 인덱스뿐 아니라 지역·원본행·세부사업명까지 대조한 뒤 `needs_llm_rerun()` 기준으로 재사용한다.

In [9]:
df_leaf_final = df_leaf.copy()
df_leaf_final["연도"] = YEAR
df_leaf_final["주요내용_정제"] = df_leaf_final["주요내용"]

if RUN_LLM:
    from functools import partial

    import yaml
    from dotenv import load_dotenv
    from openai import OpenAI

    load_dotenv(PROJECT_ROOT / ".env")
    api_key = os.getenv("UPSTAGE_API_KEY")
    if not api_key:
        raise RuntimeError("UPSTAGE_API_KEY가 없습니다.")

    with (CONFIG_DIR / "llm_extraction.yaml").open(encoding="utf-8") as file:
        llm_cfg = yaml.safe_load(file)

    client = OpenAI(
        api_key=api_key,
        base_url=llm_cfg["upstage"]["base_url"],
        max_retries=2,
        timeout=60.0,
    )
    call_once = partial(call_llm_once, client=client, llm_config=llm_cfg)
    checkpoint_df, llm_summary = run_checkpointed_refinement(
        df_leaf_final,
        checkpoint_path=CHECKPOINT_PATH,
        call_once=call_once,
        max_attempts=2,
        max_workers=2,
        chunk_size=20,
        min_call_interval_seconds=1.1,
    )
    df_leaf_final["주요내용_정제"] = checkpoint_df["주요내용_정제"]
    llm_summary_record = {
        "LLM 대상": llm_summary.llm_target_rows,
        "기존 완료 재사용": llm_summary.reused_rows,
        "신규 호출": llm_summary.new_called_rows,
        "신규 성공": llm_summary.new_success_rows,
        "실패": llm_summary.failed_rows,
        "보류": llm_summary.held_rows,
        "재실행 판정": llm_summary.rerun_rows,
        "전체 leaf": llm_summary.total_rows,
    }
else:
    checkpoint_df = pd.DataFrame(index=df_leaf_final.index)
    checkpoint_df["LLM_상태"] = "미실행"
    checkpoint_df["LLM_보존위반"] = ""
    llm_summary_record = {
        "LLM 대상": int(df_leaf_final["주요내용"].notna().sum()),
        "기존 완료 재사용": 0,
        "신규 호출": 0,
        "신규 성공": 0,
        "실패": 0,
        "보류": int(df_leaf_final["주요내용"].isna().sum()),
        "재실행 판정": 0,
        "전체 leaf": len(df_leaf_final),
    }
    print("RUN_LLM=False: 결정론적 전처리·QA만 실행합니다.")

final_status_counts = checkpoint_df["LLM_상태"].value_counts().to_dict()
llm_summary_record.update(
    {
        "최종 성공": final_status_counts.get("성공", 0),
        "최종 보존위반 원문유지": final_status_counts.get("보존위반", 0),
        "최종 결측 보류": final_status_counts.get("결측", 0),
        "최종 보류": final_status_counts.get("보존위반", 0) + final_status_counts.get("결측", 0),
        "최종 실패": final_status_counts.get("실패", 0),
    }
)
pd.DataFrame([llm_summary_record]).to_csv(LLM_SUMMARY_PATH, index=False, encoding="utf-8-sig")
preservation_review_index = checkpoint_df.index[checkpoint_df["LLM_상태"].eq("보존위반")]
preservation_review = df_leaf_final.loc[
    preservation_review_index,
    ["지역", "원본행", "세부사업명", "주요내용", "주요내용_정제"],
].copy()
preservation_review["보존위반"] = checkpoint_df.loc[
    preservation_review_index, "LLM_보존위반"
]
preservation_review["판정"] = "원문 유지"
preservation_review.to_csv(PRESERVATION_REVIEW_PATH, index=False, encoding="utf-8-sig")

if llm_summary_record["실패"]:
    failure_types = checkpoint_df.loc[
        checkpoint_df["LLM_상태"].eq("실패"), "LLM_오류유형"
    ].value_counts().to_dict()
    raise RuntimeError(f"지속적인 LLM 오류 {llm_summary_record['실패']}건: {failure_types}")

df_leaf_final["숫자보존"] = df_leaf_final.apply(
    lambda row: numbers_preserved(row["주요내용"], row["주요내용_정제"]),
    axis=1,
)
df_leaf_final["보존검사"] = df_leaf_final.apply(
    lambda row: not preservation_violations(
        row["주요내용"],
        row["주요내용_정제"],
        context_terms=(str(row["세부사업명"]),),
    ),
    axis=1,
)
if not df_leaf_final["숫자보존"].all() or not df_leaf_final["보존검사"].all():
    raise ValueError("LLM 적용 결과에 숫자·금액·퍼센트·범위·고유명사 보존 위반이 남았습니다.")

print("LLM 실행 집계:", llm_summary_record)
print("최종 보존 검증:", int(df_leaf_final["보존검사"].sum()), "건 통과")

LLM 실행 집계: {'LLM 대상': 6849, '기존 완료 재사용': 6858, '신규 호출': 0, '신규 성공': 0, '실패': 0, '보류': 0, '재실행 판정': 0, '전체 leaf': 6858, '최종 성공': 6828, '최종 보존위반 원문유지': 21, '최종 결측 보류': 9, '최종 보류': 30, '최종 실패': 0}
최종 보존 검증: 6858 건 통과


## 9. 시도별 최종 저장

wide 15개 컬럼, long 15개 컬럼, 필터링 전 전체 원본을 각각 한 번만 저장한다.

In [10]:
output_cols = [
    "연도",
    "지역",
    "대분류",
    "중분류",
    "사업분류재정구분",
    "세부사업명",
    "주요내용",
    "주요내용_정제",
    "당해예산",
    "전년도예산",
    "증감액",
    "증감율",
    "원본행",
    "지원대상",
    "지원내용_상세",
]
missing_output_cols = set(output_cols).difference(df_leaf_final.columns)
if missing_output_cols:
    raise KeyError(f"최종 저장 전에 필요한 컬럼을 확인하세요: {sorted(missing_output_cols)}")

nonempty_original = df_leaf_final["주요내용"].notna() & df_leaf_final["주요내용"].astype("string").str.strip().ne("")
empty_refined = df_leaf_final["주요내용_정제"].isna() | df_leaf_final["주요내용_정제"].astype("string").str.strip().eq("")
if (nonempty_original & empty_refined).any():
    raise ValueError("비어 있지 않은 원문에 빈 LLM 정제 결과가 있습니다.")
if df_leaf_final.duplicated(["지역", "원본행"]).any():
    raise ValueError("최종 wide 데이터에 지역·원본행 중복이 있습니다.")

wide_paths = []
long_paths = []
raw_paths = []

for sido, group in df_leaf_final.groupby("지역"):
    sido_dir = get_sido_dir(INTERIM_DIR, sido)
    output_path = sido_dir / f"{YEAR}_{sido}_세부사업_정제.csv"
    group[output_cols].to_csv(output_path, index=False, encoding="utf-8-sig")
    wide_paths.append(output_path)

    raw_path = sido_dir / f"{YEAR}_{sido}_필터링전_전체원본.csv"
    df_labeled.loc[df_labeled["지역"].eq(sido)].to_csv(
        raw_path, index=False, encoding="utf-8-sig"
    )
    raw_paths.append(raw_path)
    print(f"{sido}: {len(group)}행 저장 -> {output_path}")

id_cols = [
    "연도",
    "지역",
    "대분류",
    "중분류",
    "사업분류재정구분",
    "세부사업명",
    "주요내용",
    "증감액",
    "증감율",
    "원본행",
    "지원대상",
    "지원내용_상세",
    "주요내용_정제",
]
df_long = df_leaf_final.melt(
    id_vars=id_cols,
    value_vars=["당해예산", "전년도예산"],
    var_name="예산구분",
    value_name="예산액",
)
previous_budget_mask = df_long["예산구분"].eq("전년도예산")
df_long.loc[previous_budget_mask, "연도"] -= 1
df_long.loc[previous_budget_mask, ["증감액", "증감율"]] = None
df_long = df_long.sort_values(["지역", "원본행", "예산구분"]).reset_index(drop=True)

for sido, group in df_long.groupby("지역"):
    sido_dir = get_sido_dir(INTERIM_DIR, sido)
    long_path = sido_dir / f"{YEAR}_{sido}_세부사업_정제_long.csv"
    group.to_csv(long_path, index=False, encoding="utf-8-sig")
    long_paths.append(long_path)

if len(df_leaf_final) != 6858 or df_leaf_final["지역"].nunique() != 17:
    raise ValueError("wide 행 수 또는 지역 수가 2020년 기준과 다릅니다.")
if list(df_leaf_final[output_cols].columns) != output_cols or len(output_cols) != 15:
    raise ValueError("wide 공통 15개 컬럼을 확인하세요.")
if len(df_long) != 13716 or len(df_long.columns) != 15:
    raise ValueError("long 행 수 또는 15개 컬럼을 확인하세요.")
if not (len(wide_paths) == len(long_paths) == len(raw_paths) == 17):
    raise ValueError("시도별 산출물 파일 수가 17개가 아닙니다.")

all_output_paths = [
    *wide_paths,
    *long_paths,
    *raw_paths,
    QA_PATH,
    ROW_TYPE_REVIEW_PATH,
    LABEL_PROPAGATION_PATH,
    LLM_SUMMARY_PATH,
    PRESERVATION_REVIEW_PATH,
]
non_bom_paths = [
    path for path in all_output_paths if not path.read_bytes().startswith(b"\xef\xbb\xbf")
]
if non_bom_paths:
    raise ValueError(f"UTF-8-SIG가 아닌 산출물: {non_bom_paths}")

wide_check = pd.concat([pd.read_csv(path, encoding="utf-8-sig") for path in wide_paths])
long_check = pd.concat([pd.read_csv(path, encoding="utf-8-sig") for path in long_paths])
if len(wide_check) != len(df_leaf_final) or list(wide_check.columns) != output_cols:
    raise ValueError("저장 후 wide 행 수·CSV 스키마 검증에 실패했습니다.")
if len(long_check) != len(df_long) or list(long_check.columns) != list(df_long.columns):
    raise ValueError("저장 후 long 행 수·CSV 스키마 검증에 실패했습니다.")
if wide_check.duplicated(["지역", "원본행"]).any():
    raise ValueError("저장된 wide CSV에 중복 행이 있습니다.")
if long_check.duplicated(["지역", "원본행", "예산구분"]).any():
    raise ValueError("저장된 long CSV에 중복 행이 있습니다.")
object_text = wide_check.select_dtypes(include=["object"]).fillna("").astype(str)
if object_text.apply(lambda column: column.str.contains("�", regex=False).any()).any():
    raise ValueError("저장된 CSV에서 인코딩 대체 문자가 발견됐습니다.")
if hashlib.sha256(SOURCE_FILE.read_bytes()).hexdigest() != SOURCE_SHA256_BEFORE:
    raise ValueError("원본 XLSX 해시가 실행 중 변경됐습니다.")
if RUN_LLM and (
    llm_summary_record["기존 완료 재사용"]
    + llm_summary_record["신규 성공"]
    + llm_summary_record["실패"]
    + llm_summary_record["보류"]
    != len(df_leaf_final)
):
    raise ValueError("LLM 재사용·성공·실패·보류 집계가 전체 leaf 행 수와 다릅니다.")

print("wide:", len(wide_paths), "개 /", len(df_leaf_final), "행")
print("long:", len(long_paths), "개 /", len(df_long), "행")
print("필터링 전 원본:", len(raw_paths), "개 /", len(df_labeled), "행")
print("최종 검증: 행 수·숫자/고유명사 보존·빈 결과·중복·인코딩·스키마·원본 해시 통과")
print("실행 오류: 0건")

강원: 470행 저장 -> D:\University\yumocha\yumocha-review-2020\data\interim\강원\2020_강원_세부사업_정제.csv
경기: 1010행 저장 -> D:\University\yumocha\yumocha-review-2020\data\interim\경기\2020_경기_세부사업_정제.csv
경남: 657행 저장 -> D:\University\yumocha\yumocha-review-2020\data\interim\경남\2020_경남_세부사업_정제.csv
경북: 618행 저장 -> D:\University\yumocha\yumocha-review-2020\data\interim\경북\2020_경북_세부사업_정제.csv
광주: 179행 저장 -> D:\University\yumocha\yumocha-review-2020\data\interim\광주\2020_광주_세부사업_정제.csv
대구: 238행 저장 -> D:\University\yumocha\yumocha-review-2020\data\interim\대구\2020_대구_세부사업_정제.csv
대전: 299행 저장 -> D:\University\yumocha\yumocha-review-2020\data\interim\대전\2020_대전_세부사업_정제.csv
부산: 545행 저장 -> D:\University\yumocha\yumocha-review-2020\data\interim\부산\2020_부산_세부사업_정제.csv
서울: 177행 저장 -> D:\University\yumocha\yumocha-review-2020\data\interim\서울\2020_서울_세부사업_정제.csv
세종: 107행 저장 -> D:\University\yumocha\yumocha-review-2020\data\interim\세종\2020_세종_세부사업_정제.csv
울산: 194행 저장 -> D:\University\yumocha\yumocha-review-2020\data\interim

wide: 17 개 / 6858 행
long: 17 개 / 13716 행
필터링 전 원본: 17 개 / 7003 행
최종 검증: 행 수·숫자/고유명사 보존·빈 결과·중복·인코딩·스키마·원본 해시 통과
실행 오류: 0건


C:\Users\JunGu\AppData\Local\Temp\ipykernel_17664\3366797474.py:113: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_text = wide_check.select_dtypes(include=["object"]).fillna("").astype(str)


## 완료 전 체크리스트

- [x] 17개 시도 포함 여부 확인
- [x] 행 유형 분포 확인
- [x] 연도 고유 헤더·연속행 후보 원본 대조
- [x] 재원구분(계/국비/지방비) 이중계상 여부 확인 및 `NEEDS_FUNDING_SOURCE_CLEANUP` 반영
- [x] 중분류 소계 QA 불일치 원인·보정액·보정근거 기록
- [x] 경남 7494 뒤 중분류 라벨 142행 검증
- [x] 증감액·증감율 재계산 결과 확인
- [x] LLM 숫자·금액·퍼센트·범위·고유명사 보존 검사
- [x] wide/long 행 수와 컬럼 스키마 확인
- [x] 최종 CSV `utf-8-sig` 저장 확인
- [x] 완료 체크포인트 재사용 전체 실행
- [x] 리팩터링 전후 QA 수치 비교